# Build Municipality Output Tables

This notebook builds municipality-level analytical tables from the cleaned processed datasets.

It starts with the recovery service access table, then will later join overdose burden, EMS burden, and vulnerability indicators.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
processed_dir = DATA_DIR / "processed"

print("Processed folder exists:", processed_dir.exists())

In [ ]:
import geopandas as gpd

towns_lookup_path = processed_dir / "ma_municipalities_lookup.csv"

if towns_lookup_path.exists():
    towns = pd.read_csv(towns_lookup_path)
    print("Loaded existing municipality lookup:", len(towns))
else:
    towns_geo_path = processed_dir / "ma_municipalities_clean.geojson"
    towns_geo = gpd.read_file(towns_geo_path)

    towns = towns_geo[["TOWN", "TOWN_ID", "COUNTY"]].copy()
    towns["town_join"] = towns["TOWN"].str.upper().str.strip()

    towns.to_csv(towns_lookup_path, index=False)
    print("Created municipality lookup:", len(towns))

print(towns_lookup_path.exists())
towns.head()

## Load cleaned service datasets

In [ ]:
samhsa = pd.read_csv(processed_dir / "samhsa_treatment_facilities_clean.csv", dtype={"zip": "string"})
peer = pd.read_csv(processed_dir / "peer_recovery_centers_clean.csv", dtype={"zip": "string"})
ssp = pd.read_csv(processed_dir / "syringe_service_programs_clean.csv", dtype={"zip": "string"})
harm = pd.read_csv(processed_dir / "harm_reduction_programs_deduped_clean.csv", dtype={"zip": "string"})

print("Municipalities:", len(towns))
print("SAMHSA treatment facilities:", len(samhsa))
print("Peer recovery centers:", len(peer))
print("Syringe service programs:", len(ssp))
print("Harm reduction programs:", len(harm))

## Aggregate service counts by municipality

In [ ]:
# Standardize town join keys

towns_access = towns[["TOWN", "TOWN_ID", "COUNTY", "town_join"]].copy()

samhsa["town_join"] = samhsa["city_join"].str.upper().str.strip()
peer["town_join"] = peer["city_join"].str.upper().str.strip()
ssp["town_join"] = ssp["city_join"].str.upper().str.strip()
harm["town_join"] = harm["city_join"].str.upper().str.strip()

treatment_counts = (
    samhsa.groupby("town_join")
    .size()
    .reset_index(name="treatment_facility_count")
)

peer_counts = (
    peer.groupby("town_join")
    .size()
    .reset_index(name="peer_recovery_count")
)

ssp_counts = (
    ssp.groupby("town_join")
    .size()
    .reset_index(name="ssp_count")
)

harm_counts = (
    harm.groupby("town_join")
    .size()
    .reset_index(name="harm_reduction_count")
)

print("Towns with treatment:", len(treatment_counts))
print("Towns with peer recovery:", len(peer_counts))
print("Towns with SSP:", len(ssp_counts))
print("Towns with harm reduction:", len(harm_counts))

In [ ]:
access = towns_access.merge(treatment_counts, on="town_join", how="left")
access = access.merge(peer_counts, on="town_join", how="left")
access = access.merge(ssp_counts, on="town_join", how="left")
access = access.merge(harm_counts, on="town_join", how="left")

count_cols = [
    "treatment_facility_count",
    "peer_recovery_count",
    "ssp_count",
    "harm_reduction_count",
]

access[count_cols] = access[count_cols].fillna(0).astype(int)

print("Access table rows:", len(access))
print("Expected municipalities:", len(towns_access))

access.head()

In [ ]:
access["has_treatment"] = (access["treatment_facility_count"] > 0).astype(int)
access["has_peer_recovery"] = (access["peer_recovery_count"] > 0).astype(int)
access["has_ssp"] = (access["ssp_count"] > 0).astype(int)
access["has_harm_reduction"] = (access["harm_reduction_count"] > 0).astype(int)

access["service_diversity_score"] = (
    access["has_treatment"]
    + access["has_peer_recovery"]
    + access["has_ssp"]
    + access["has_harm_reduction"]
)

print("Towns with treatment:", access["has_treatment"].sum())
print("Towns with peer recovery:", access["has_peer_recovery"].sum())
print("Towns with SSP:", access["has_ssp"].sum())
print("Towns with harm reduction:", access["has_harm_reduction"].sum())

access["service_diversity_score"].value_counts().sort_index()

In [ ]:
summary_cols = [
    "TOWN",
    "COUNTY",
    "treatment_facility_count",
    "peer_recovery_count",
    "ssp_count",
    "harm_reduction_count",
    "has_treatment",
    "has_peer_recovery",
    "has_ssp",
    "has_harm_reduction",
    "service_diversity_score",
]

access_summary = access[summary_cols].copy()

access_summary.sort_values(
    ["service_diversity_score", "harm_reduction_count", "treatment_facility_count"],
    ascending=False
).head(25)

In [ ]:
access_csv_path = processed_dir / "municipality_recovery_access_table.csv"

access_summary.to_csv(access_csv_path, index=False)

print("Saved:", access_csv_path)
print(access_csv_path.exists())
print("Rows:", len(access_summary))
print("Columns:", access_summary.columns.tolist())

## Check service towns that did not match municipalities

In [ ]:
def unmatched_service_towns(service_df, service_name):
    service_towns = set(service_df["town_join"].dropna().unique())
    official_towns = set(towns_access["town_join"].dropna().unique())
    unmatched = sorted(service_towns - official_towns)

    print(f"{service_name} unmatched town names:", len(unmatched))
    return unmatched

unmatched_samhsa = unmatched_service_towns(samhsa, "SAMHSA")
unmatched_peer = unmatched_service_towns(peer, "Peer recovery")
unmatched_ssp = unmatched_service_towns(ssp, "SSP")
unmatched_harm = unmatched_service_towns(harm, "Harm reduction")

unmatched_samhsa[:30], unmatched_peer[:30], unmatched_ssp[:30], unmatched_harm[:30]

In [ ]:
town_crosswalk = {
    # Boston neighborhoods
    "ALLSTON": "BOSTON",
    "BRIGHTON": "BOSTON",
    "CHARLESTOWN": "BOSTON",
    "DORCHESTER": "BOSTON",
    "DORCHESTER CENTER": "BOSTON",
    "JAMAICA PLAIN": "BOSTON",
    "MATTAPAN": "BOSTON",
    "ROSLINDALE": "BOSTON",
    "ROXBURY": "BOSTON",
    "SOUTH BOSTON": "BOSTON",
    "WEST ROXBURY": "BOSTON",
    "EAST BOSTON": "BOSTON",

    # Barnstable villages
    "HYANNIS": "BARNSTABLE",
    "CENTERVILLE": "BARNSTABLE",
    "OSTERVILLE": "BARNSTABLE",
    "WEST BARNSTABLE": "BARNSTABLE",
    "BARNSTABLE VILLAGE": "BARNSTABLE",

    # Other common villages/localities
    "BUZZARDS BAY": "BOURNE",
    "EAST FALMOUTH": "FALMOUTH",
    "EAST WAREHAM": "WAREHAM",
    "EAST WEYMOUTH": "WEYMOUTH",
    "FEEDING HILLS": "AGAWAM",
    "INDIAN ORCHARD": "SPRINGFIELD",
    "INDIAN ORCHARDS": "SPRINGFIELD",
    "LEEDS": "NORTHAMPTON",
    "BALDWINVILLE": "TEMPLETON",
    "CHESTNUT HILL": "NEWTON",
}

In [ ]:
def apply_town_crosswalk(df):
    df = df.copy()
    df["town_join_original"] = df["town_join"]
    df["town_join"] = df["town_join"].replace(town_crosswalk)
    return df

samhsa = apply_town_crosswalk(samhsa)
peer = apply_town_crosswalk(peer)
ssp = apply_town_crosswalk(ssp)
harm = apply_town_crosswalk(harm)

In [ ]:
unmatched_samhsa = unmatched_service_towns(samhsa, "SAMHSA")
unmatched_peer = unmatched_service_towns(peer, "Peer recovery")
unmatched_ssp = unmatched_service_towns(ssp, "SSP")
unmatched_harm = unmatched_service_towns(harm, "Harm reduction")

unmatched_samhsa[:50], unmatched_peer[:50], unmatched_ssp[:50], unmatched_harm[:50]

In [ ]:
town_crosswalk.update({
    "NEEDHAM HEIGHTS": "NEEDHAM",
    "NEWTON UPPER FALLS": "NEWTON",
    "NORTH BILLERICA": "BILLERICA",
    "NORTH DARTMOUTH": "DARTMOUTH",
    "ROCHDALE": "LEICESTER",
    "SOUTH WEYMOUTH": "WEYMOUTH",
    "VINEYARD HAVEN": "TISBURY",
    "WHITINSVILLE": "NORTHBRIDGE",
})

In [ ]:
samhsa = apply_town_crosswalk(samhsa)
peer = apply_town_crosswalk(peer)
ssp = apply_town_crosswalk(ssp)
harm = apply_town_crosswalk(harm)

In [ ]:
unmatched_harm = unmatched_service_towns(harm, "Harm reduction")

harm[harm["town_join"].isin(unmatched_harm)][
    ["address", "city", "zip", "town_join"]
].head(50)

In [ ]:
zip_to_town = {
    "01880": "WAKEFIELD",
    "01569": "UXBRIDGE",
    "01876": "TEWKSBURY",
    "01151": "SPRINGFIELD",
    "01105": "SPRINGFIELD",
    "01108": "SPRINGFIELD",
    "01107": "SPRINGFIELD",
    "01109": "SPRINGFIELD",
    "02128": "BOSTON",
    "02124": "BOSTON",
    "01863": "CHELMSFORD",
    "01258": "EGREMONT",
    "01201": "PITTSFIELD",
    "02191": "WEYMOUTH",
    "01778": "WAYLAND",
    "01752": "MARLBOROUGH",
    "02180": "STONEHAM",
    "02067": "SHARON",
    "02115": "BOSTON",
    "02720": "FALL RIVER",
    "02724": "FALL RIVER",
    "01960": "PEABODY",
    "01040": "HOLYOKE",
    "02346": "MIDDLEBOROUGH",
    "02169": "QUINCY",
    "02780": "TAUNTON",
    "01901": "LYNN",
    "02301": "BROCKTON",
    "01742": "CONCORD",
    "01364": "ORANGE",
    "01950": "NEWBURYPORT",
    "02118": "BOSTON",
    "01930": "GLOUCESTER",
    "02351": "ABINGTON",
}

# Apply ZIP-based repair only where town_join is not an official MA municipality
official_towns = set(towns_access["town_join"].dropna().unique())

bad_harm_mask = ~harm["town_join"].isin(official_towns)

harm.loc[bad_harm_mask, "town_join"] = (
    harm.loc[bad_harm_mask, "zip"]
    .astype(str)
    .str[:5]
    .map(zip_to_town)
    .fillna(harm.loc[bad_harm_mask, "town_join"])
)

# Keep city aligned with town_join for repaired rows
harm.loc[bad_harm_mask & harm["zip"].astype(str).str[:5].isin(zip_to_town.keys()), "city"] = (
    harm.loc[bad_harm_mask & harm["zip"].astype(str).str[:5].isin(zip_to_town.keys()), "zip"]
    .astype(str)
    .str[:5]
    .map(zip_to_town)
    .str.title()
)

unmatched_harm_after_zip = sorted(set(harm["town_join"].dropna().unique()) - official_towns)

print("Remaining unmatched harm town names:", len(unmatched_harm_after_zip))
unmatched_harm_after_zip[:50]

## Rebuild service counts after fixes

In [ ]:
treatment_counts = (
    samhsa.groupby("town_join")
    .size()
    .reset_index(name="treatment_facility_count")
)

peer_counts = (
    peer.groupby("town_join")
    .size()
    .reset_index(name="peer_recovery_count")
)

ssp_counts = (
    ssp.groupby("town_join")
    .size()
    .reset_index(name="ssp_count")
)

harm_counts = (
    harm.groupby("town_join")
    .size()
    .reset_index(name="harm_reduction_count")
)

print("Towns with treatment:", len(treatment_counts))
print("Towns with peer recovery:", len(peer_counts))
print("Towns with SSP:", len(ssp_counts))
print("Towns with harm reduction:", len(harm_counts))

In [ ]:
access = towns_access.merge(treatment_counts, on="town_join", how="left")
access = access.merge(peer_counts, on="town_join", how="left")
access = access.merge(ssp_counts, on="town_join", how="left")
access = access.merge(harm_counts, on="town_join", how="left")

count_cols = [
    "treatment_facility_count",
    "peer_recovery_count",
    "ssp_count",
    "harm_reduction_count",
]

access[count_cols] = access[count_cols].fillna(0).astype(int)

access["has_treatment"] = (access["treatment_facility_count"] > 0).astype(int)
access["has_peer_recovery"] = (access["peer_recovery_count"] > 0).astype(int)
access["has_ssp"] = (access["ssp_count"] > 0).astype(int)
access["has_harm_reduction"] = (access["harm_reduction_count"] > 0).astype(int)

access["service_diversity_score"] = (
    access["has_treatment"]
    + access["has_peer_recovery"]
    + access["has_ssp"]
    + access["has_harm_reduction"]
)

print("Access table rows:", len(access))
print("Towns with treatment:", access["has_treatment"].sum())
print("Towns with peer recovery:", access["has_peer_recovery"].sum())
print("Towns with SSP:", access["has_ssp"].sum())
print("Towns with harm reduction:", access["has_harm_reduction"].sum())

access["service_diversity_score"].value_counts().sort_index()

In [ ]:
summary_cols = [
    "TOWN",
    "TOWN_ID",
    "COUNTY",
    "town_join",
    "treatment_facility_count",
    "peer_recovery_count",
    "ssp_count",
    "harm_reduction_count",
    "has_treatment",
    "has_peer_recovery",
    "has_ssp",
    "has_harm_reduction",
    "service_diversity_score",
]

access_summary = access[summary_cols].copy()

access_csv_path = processed_dir / "municipality_recovery_access_table.csv"
access_summary.to_csv(access_csv_path, index=False)

print("Saved:", access_csv_path)
print(access_csv_path.exists())
print("Rows:", len(access_summary))
print("Columns:", len(access_summary.columns))

access_summary.sort_values(
    ["service_diversity_score", "harm_reduction_count", "treatment_facility_count"],
    ascending=False
).head(20)